## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [13]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)

import pprint


## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [ ]:

# Make an agent with name, instructions, model
# instructions are the system prompt, model is the model to use

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [17]:
# Run the joke with Runner.run(agent, prompt)
# prompt is the user prompt, which is the input to the agent
result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents") # returns RunResult object
pprint.pprint(result)



RunResult(input='Tell a joke about Autonomous AI Agents',
          new_items=[MessageOutputItem(agent=Agent(name='Jokester',
                                                   handoff_description=None,
                                                   tools=[],
                                                   mcp_servers=[],
                                                   mcp_config={},
                                                   instructions='You are a '
                                                                'joke teller',
                                                   prompt=None,
                                                   handoffs=[],
                                                   model='gpt-5.4-mini',
                                                   model_settings=ModelSettings(temperature=None,
                                                                                top_p=None,
                                                        

In [18]:
# Here is the final output
pprint.pprint(result.final_output)

('Autonomous AI agents are like interns who never sleep, never eat, and never '
 'ask for help…\n'
 '\n'
 'which is great, until they confidently organize your entire life around a '
 '“small optimization” you never requested.')


In [22]:
# Here is the detail of the LLM calls
pprint.pprint(result.to_input_list()) # returns a list of RunInput objects, which contain the input and output of each LLM call

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'content': [{'annotations': [],
               'logprobs': [],
               'text': 'Autonomous AI agents are like interns who never sleep, '
                       'never eat, and never ask for help…\n'
                       '\n'
                       'which is great, until they confidently organize your '
                       'entire life around a “small optimization” you never '
                       'requested.',
               'type': 'output_text'}],
  'id': 'msg_014ec70010845e93006aa44559249c87d2bfe10710eff9fbcd',
  'phase': 'final_answer',
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]


## Adding Observability with a trace

In [39]:
# observe the trace of the run, which shows the steps taken by the agent to produce the final output
# trace will logs the steps to OpenAI API platform, and you can view the trace in the OpenAI platform under the "Traces" tab

with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")

In [42]:
print(result.final_output)

Autonomous AI agents are just like interns: they say, “I’ve got this,” then make three decisions, spawn a sub-agent, and somehow spend all day optimizing the wrong thing.


## Now go and look at the trace

https://platform.openai.com/traces

In [50]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.") # returns RunResultStreaming object
pprint.pprint(result)


# stream_events() returns an async generator of events, which can be used to stream the output of the agent in real-time
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

RunResultStreaming(input='Please tell me 5 jokes about AI Agents.',
                   new_items=[],
                   raw_responses=[],
                   final_output=None,
                   input_guardrail_results=[],
                   output_guardrail_results=[],
                   tool_input_guardrail_results=[],
                   tool_output_guardrail_results=[],
                   context_wrapper=RunContextWrapper(context=None,
                                                     usage=Usage(requests=0,
                                                                 input_tokens=0,
                                                                 input_tokens_details=InputTokensDetails(cached_tokens=0),
                                                                 output_tokens=0,
                                                                 output_tokens_details=OutputTokensDetails(reasoning_tokens=0),
                                                                 t

## Part 2: Adding a tool

In [51]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [52]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [53]:
push("HEY!!")

Push: HEY!!


In [54]:
push

<function __main__.push(message)>

In [55]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [57]:
pprint.pprint(push_tool)

FunctionTool(name='push_tool',
             description='Send the given message to the user as a push '
                         'notification',
             params_json_schema={'additionalProperties': False,
                                 'properties': {'message': {'title': 'Message',
                                                            'type': 'string'}},
                                 'required': ['message'],
                                 'title': 'push_tool_args',
                                 'type': 'object'},
             on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001C1378D71D0>,
             strict_json_schema=True,
             is_enabled=True,
             tool_input_guardrails=None,
             tool_output_guardrails=None,
             needs_approval=False,
             timeout_seconds=None,
             timeout_behavior='error_as_result',
             timeout_error_function=None,
             defer_loading=False)


In [58]:
push_tool.description

'Send the given message to the user as a push notification'

In [59]:
notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])

In [60]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


Done.


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

**But each call to Runner.run() is a fresh start.**

Let's see that:

In [61]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [62]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed — nice to meet you! How can I help you today?


In [63]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don’t know your name from this chat alone. If you’d like, tell me your name and I’ll use it.


## Memory approach 1 - just manually pass in the list of dicts

In [64]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed — nice to meet you! How can I help today?


In [65]:
response.to_input_list()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_09305e74e29e009f006aa45938a2fc87d2b8d7eb0b855605ff',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you! How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [66]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_09305e74e29e009f006aa45938a2fc87d2b8d7eb0b855605ff',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you! How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name?"}]

In [67]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Ed.


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [68]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [72]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

Hi Ed — nice to meet you! How can I help today?


In [76]:
await session.get_items()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_08ca4f74faffd96e006aa4597bef5087d2ba6b0e13e3b7dc8c',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you! How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [77]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Ed.


In [79]:
await session.get_items()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_08ca4f74faffd96e006aa4597bef5087d2ba6b0e13e3b7dc8c',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you! How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'content': "What's my name?", 'role': 'user'},
 {'id': 'msg_08ca4f74faffd96e006aa459a96dc487d2b58419277752444c',
  'content': [{'annotations': [],
    'text': 'Your name is Ed.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

Clearing the session memory

In [80]:
await session.clear_session()

In [81]:
await session.get_items()

[]

In [82]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

I don’t know your name unless you tell me. If you want, share it and I’ll remember it for this conversation.


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>